# Qwen full LLM multi-agent TEC experiment

This notebook runs the full LLM-driven multi-agent network on the same five TEC tasks used by the Qwen single-agent batch notebook.

## 1. Clean Colab setup

Run this in Google Colab first. After it finishes, restart the runtime before importing transformers. This notebook expects an already processed TEC parquet and does not rebuild IONEX data.

In [ ]:
# Clean Colab setup for Qwen.
# Run this cell first, then restart runtime before importing transformers.

!pip uninstall -y transformers scikit-learn scipy
!pip install -U accelerate bitsandbytes torchvision pillow sentencepiece protobuf safetensors huggingface_hub pandas pyarrow pydantic python-dateutil
!pip install "transformers @ git+https://github.com/huggingface/transformers.git@main"

IMPORTANT:
After this cell finishes, restart the Colab runtime:
Runtime -> Restart runtime
Do not import transformers before restarting.

## 2. Import check

In [ ]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print("transformers Qwen imports OK")

## 3. Clone or update repository

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/tec_agent_project")
REPO_URL = "https://github.com/ilyakosilov/tec_agent_project.git"

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", "main"], check=True)

os.chdir(PROJECT_DIR)

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Working directory:", Path.cwd())
print("Latest commit:")
subprocess.run(["git", "log", "--oneline", "-3"], check=False)

## 4. Project imports

In [ ]:
import json
from pathlib import Path

import pandas as pd

from tec_agents.agents.llm_multi_agent import LLMFullMultiAgent
from tec_agents.data.datasets import get_dataset_summary, register_dataset
from tec_agents.eval.five_task_configs import (
    get_five_task_configs,
    build_five_task_eval_task,
    build_five_task_expected_sequence,
)
from tec_agents.eval.gold_runner import GoldRunner
from tec_agents.eval.metrics import MetricResult, compare_agent_to_gold
from tec_agents.eval.task_set import task_to_dict
from tec_agents.llm.local_qwen import LocalQwenChatModel
from tec_agents.mcp.client import LocalMCPClient
from tec_agents.mcp.server import build_local_mcp_server

## 5. CONFIG

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
TORCH_DTYPE = "float16"
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
DO_SAMPLE = False

DATASET_REF = "default"
DATASET_FILENAME = "tec_regions_2024_01_01_to_2024_04_01_hourly.parquet"
DATASET_PATH = PROJECT_DIR / "data" / "processed" / DATASET_FILENAME
START = "2024-03-01"
END = "2024-04-01"

ARCHITECTURE_MODE = "qwen_multi_agent_full_llm"
STATE_FEEDBACK_MODE = "state_aware"
MAX_ORCHESTRATION_STEPS = 12
MAX_ROLE_STEPS = 8
MAX_TOOL_CALLS_PER_ROLE = 8
MAX_PARSE_RETRIES = 2
MAX_TOOL_RETRIES = 2

RUN_ALL_TASKS = True
SELECTED_PRESET = "hightec_midlat_europe"

OUTPUT_DIR = PROJECT_DIR / "outputs" / "metrics"
BATCH_OUTPUT_PATH = OUTPUT_DIR / "qwen_multi_agent_batch_colab.json"
PER_TASK_OUTPUT_TEMPLATE = "qwen_multi_agent_{preset_id}_colab.json"

TASK_CONFIGS = get_five_task_configs(dataset_ref=DATASET_REF)
for config in TASK_CONFIGS:
    config["start"] = START
    config["end"] = END

if RUN_ALL_TASKS:
    SELECTED_TASK_CONFIGS = list(TASK_CONFIGS)
else:
    SELECTED_TASK_CONFIGS = [
        item for item in TASK_CONFIGS if item["preset_id"] == SELECTED_PRESET
    ]
    assert SELECTED_TASK_CONFIGS, f"Unknown SELECTED_PRESET: {SELECTED_PRESET}"

print("architecture:", ARCHITECTURE_MODE)
print("RUN_ALL_TASKS:", RUN_ALL_TASKS)
print("selected presets:", [item["preset_id"] for item in SELECTED_TASK_CONFIGS])

## Planned test questions

This preview is for the human notebook reader only. It is not passed into LLM prompts.

In [ ]:
preview_rows = []
for config in SELECTED_TASK_CONFIGS:
    preview_rows.append(
        {
            "preset_id": config["preset_id"],
            "task_type": config["task_type"],
            "query": config["query"],
            "expected_tool_sequence": " -> ".join(build_five_task_expected_sequence(config)),
        }
    )

pd.DataFrame(preview_rows)

## 6. Dataset setup

Required processed dataset: `data/processed/tec_regions_2024_01_01_to_2024_04_01_hourly.parquet`. This notebook does not rebuild IONEX data.

In [ ]:
DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
print("Expected dataset:", DATASET_PATH)
print("Exists:", DATASET_PATH.exists())

# Optional Google Drive copy:
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_PARQUET_PATH = Path("/content/drive/MyDrive/tec_regions_2024_01_01_to_2024_04_01_hourly.parquet")
# import shutil
# shutil.copy2(DRIVE_PARQUET_PATH, DATASET_PATH)

assert DATASET_PATH.exists(), f"Missing dataset: {DATASET_PATH}"
register_dataset(dataset_ref=DATASET_REF, path=DATASET_PATH, file_format="parquet")
get_dataset_summary(DATASET_REF)

## 7. Optional Hugging Face login

In [ ]:
# Optional: use a Hugging Face token for higher rate limits or gated models.
try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN") or userdata.get("HF_read_token")
    if hf_token:
        login(token=hf_token)
        print("Logged in to Hugging Face.")
    else:
        print("No HF token found. Continuing unauthenticated.")
except Exception as exc:
    print("HF login skipped:", exc)

## 8. Load Qwen

In [ ]:
model = LocalQwenChatModel(
    model_name=MODEL_NAME,
    device_map="auto",
    torch_dtype=TORCH_DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=LOAD_IN_8BIT,
    trust_remote_code=True,
    max_input_tokens=MAX_INPUT_TOKENS,
)

## 9. Helper functions

In [ ]:
def to_jsonable(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if hasattr(value, "to_dict"):
        return to_jsonable(value.to_dict())
    if hasattr(value, "__dataclass_fields__"):
        return {field: to_jsonable(getattr(value, field)) for field in value.__dataclass_fields__}
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_jsonable(item) for item in value]
    return str(value)


def actual_tool_sequence(result):
    return [call.get("tool_name") for call in (result.trace or {}).get("calls", [])]


def agent_result_for_metrics(result):
    payload = dict(result.tool_results or {})
    payload["answer"] = result.answer
    payload["final_answer"] = result.answer
    return payload


def build_verdict_checks(result, metric_result):
    metrics = metric_result.metrics if metric_result else {}
    return {
        "agent_success": bool(result.success),
        "metric_success": bool(metric_result and metric_result.success),
        "final_answer_present": bool(result.answer),
        "tool_sequence_match": metrics.get("tool_sequence_match") is True,
        "no_tool_errors": metrics.get("tool_error_count") == 0,
        "role_agent_order_match": metrics.get("role_agent_order_match") is True,
        "required_role_agents_called": metrics.get("required_role_agents_called") is True,
        "artifact_flow_valid": metrics.get("artifact_flow_valid") is True,
        "no_forbidden_tools": result.forbidden_tool_call_count == 0,
        "no_stall": result.stalled_loop_detected is False,
        "no_legacy_report_tool_used": metrics.get("legacy_report_tool_used") is not True,
    }


def key_metric_summary(task_type, metrics):
    if task_type == "high_tec":
        return f"thr_err={metrics.get('threshold_abs_error')}; interval_err={metrics.get('interval_count_error')}; peak_err={metrics.get('global_peak_abs_error')}"
    if task_type == "stable_intervals":
        return f"stable_count_err={metrics.get('stable_interval_count_error')}; top_duration_err={metrics.get('top_stable_duration_abs_error')}"
    if task_type == "compare_regions":
        return f"mean_err_avg={metrics.get('mean_abs_error_avg')}; p90_err_avg={metrics.get('p90_abs_error_avg')}; pairwise_match={metrics.get('pairwise_delta_count_match')}"
    if task_type == "report":
        return f"required_artifacts={metrics.get('required_artifacts_present')}; grounded={metrics.get('report_grounded_in_artifacts')}; high_count_err={metrics.get('report_high_tec_interval_count_error_avg')}; stable_count_err={metrics.get('report_stable_interval_count_error_avg')}"
    return ""

## 10. Run full LLM multi-agent batch

In [ ]:
all_results = []
gold_runner = GoldRunner()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for index, task_config in enumerate(SELECTED_TASK_CONFIGS, start=1):
    preset_id = task_config["preset_id"]
    task_type = task_config["task_type"]
    query = task_config["query"]
    expected_sequence = build_five_task_expected_sequence(task_config)
    eval_task = build_five_task_eval_task(task_config)

    print("=" * 100)
    print(f"TASK {index}/{len(SELECTED_TASK_CONFIGS)}: {preset_id} ({task_type})")
    print(query)

    server = build_local_mcp_server(run_id=f"qwen_multi_agent_{preset_id}_colab")
    client = LocalMCPClient(server)
    agent = LLMFullMultiAgent(
        model=model,
        client=client,
        max_orchestration_steps=MAX_ORCHESTRATION_STEPS,
        max_role_steps=MAX_ROLE_STEPS,
        max_tool_calls_per_role=MAX_TOOL_CALLS_PER_ROLE,
        max_parse_retries=MAX_PARSE_RETRIES,
        max_tool_retries=MAX_TOOL_RETRIES,
        temperature=TEMPERATURE,
        state_feedback_mode=STATE_FEEDBACK_MODE,
    )

    result = agent.run(query)
    gold_result = gold_runner.run(eval_task)

    if gold_result.status == "success" and gold_result.result is not None:
        metric_result = compare_agent_to_gold(
            task_id=eval_task.task_id,
            task_type=eval_task.task_type,
            agent_result=agent_result_for_metrics(result),
            gold_result=gold_result.result,
            agent_trace=result.trace,
            task=task_to_dict(eval_task),
            parsed_task=result.parsed_task,
            orchestration_steps=result.orchestration_steps,
        )
    else:
        metric_result = MetricResult(
            task_id=eval_task.task_id,
            task_type=eval_task.task_type,
            success=False,
            metrics={},
            errors=[f"Gold runner failed: {gold_result.error}"],
        )

    verdict_checks = build_verdict_checks(result, metric_result)
    overall_ok = all(verdict_checks.values())
    metrics = metric_result.metrics
    record = {
        "selected_preset": preset_id,
        "task_config": task_config,
        "query": query,
        "expected_tool_sequence": expected_sequence,
        "actual_tool_sequence": actual_tool_sequence(result),
        "architecture": ARCHITECTURE_MODE,
        "model_name": MODEL_NAME,
        "agent_result": result.to_dict(),
        "gold_status": gold_result.status,
        "gold_error": gold_result.error,
        "gold_result": gold_result.result,
        "metrics": metric_result.to_dict(),
        "verdict_checks": verdict_checks,
        "overall_ok": overall_ok,
        "success": bool(result.success),
        "final_answer_preview": " ".join(result.answer.split())[:500],
        "role_agent_order": metrics.get("role_agent_order"),
        "expected_role_agent_order": metrics.get("expected_role_agent_order"),
        "role_agent_order_match": metrics.get("role_agent_order_match"),
        "handoff_count": metrics.get("handoff_count"),
        "data_agent_called": metrics.get("data_agent_called"),
        "math_agent_called": metrics.get("math_agent_called"),
        "analysis_agent_called": metrics.get("analysis_agent_called"),
        "report_agent_called": metrics.get("report_agent_called"),
        "required_role_agents_called": metrics.get("required_role_agents_called"),
        "artifact_flow_valid": metrics.get("artifact_flow_valid"),
        "retry_count": metrics.get("retry_count"),
        "recovery_attempt_count": metrics.get("recovery_attempt_count"),
        "recovery_success_count": metrics.get("recovery_success_count"),
        "recovery_failure_count": metrics.get("recovery_failure_count"),
    }
    all_results.append(record)

    per_task_path = OUTPUT_DIR / PER_TASK_OUTPUT_TEMPLATE.format(preset_id=preset_id)
    with per_task_path.open("w", encoding="utf-8") as f:
        json.dump(to_jsonable(record), f, ensure_ascii=False, indent=2)
    print("Saved:", per_task_path)
    print("agent_success:", result.success)
    print("overall_ok:", overall_ok)
    print("tool_sequence_match:", metrics.get("tool_sequence_match"))
    print("role_agent_order_match:", metrics.get("role_agent_order_match"))
    print("artifact_flow_valid:", metrics.get("artifact_flow_valid"))

## 11. Summary table and save aggregate JSON

In [ ]:
summary_rows = []
for record in all_results:
    result = record["agent_result"]
    metrics = record["metrics"]["metrics"]
    task_type = record["task_config"]["task_type"]
    summary_rows.append(
        {
            "preset_id": record["selected_preset"],
            "task_type": task_type,
            "agent_success": result["success"],
            "overall_ok": record["overall_ok"],
            "tool_sequence_match": metrics.get("tool_sequence_match"),
            "tool_call_count": metrics.get("tool_call_count"),
            "tool_error_count": metrics.get("tool_error_count"),
            "final_answer_present": bool(result["answer"]),
            "role_agent_order_match": metrics.get("role_agent_order_match"),
            "required_role_agents_called": metrics.get("required_role_agents_called"),
            "artifact_flow_valid": metrics.get("artifact_flow_valid"),
            "handoff_count": metrics.get("handoff_count"),
            "retry_count": metrics.get("retry_count"),
            "parse_error_count": result["parse_error_count"],
            "repeated_tool_call_count": result["repeated_tool_call_count"],
            "forbidden_tool_call_count": result["forbidden_tool_call_count"],
            "stalled_loop_detected": result["stalled_loop_detected"],
            "key_metric_summary": key_metric_summary(task_type, metrics),
            "answer_preview": record["final_answer_preview"],
        }
    )

summary_df = pd.DataFrame(summary_rows)

def bool_rate(values):
    values = [value for value in values if isinstance(value, bool)]
    return (sum(1 for value in values if value) / len(values)) if values else None

summary = {
    "n_tasks": len(all_results),
    "n_success": sum(1 for item in all_results if item["agent_result"]["success"]),
    "n_failed": sum(1 for item in all_results if not item["agent_result"]["success"]),
    "success_rate": bool_rate([item["agent_result"]["success"] for item in all_results]),
    "n_overall_ok": sum(1 for item in all_results if item["overall_ok"]),
    "overall_ok_rate": bool_rate([item["overall_ok"] for item in all_results]),
    "agent_success_rate": bool_rate([item["agent_result"]["success"] for item in all_results]),
    "tool_sequence_match_rate": bool_rate([item["metrics"]["metrics"].get("tool_sequence_match") for item in all_results]),
    "role_agent_order_match_rate": bool_rate([item["metrics"]["metrics"].get("role_agent_order_match") for item in all_results]),
    "artifact_flow_valid_rate": bool_rate([item["metrics"]["metrics"].get("artifact_flow_valid") for item in all_results]),
    "required_role_agents_called_rate": bool_rate([item["metrics"]["metrics"].get("required_role_agents_called") for item in all_results]),
    "avg_tool_call_count": summary_df["tool_call_count"].dropna().mean() if "tool_call_count" in summary_df else None,
    "avg_tool_error_count": summary_df["tool_error_count"].dropna().mean() if "tool_error_count" in summary_df else None,
    "avg_retry_count": summary_df["retry_count"].dropna().mean() if "retry_count" in summary_df else None,
    "avg_parse_error_count": summary_df["parse_error_count"].dropna().mean() if "parse_error_count" in summary_df else None,
    "stalled_loop_rate": bool_rate([item["agent_result"]["stalled_loop_detected"] for item in all_results]),
    "forbidden_tool_call_rate": bool_rate([item["agent_result"]["forbidden_tool_call_count"] > 0 for item in all_results]),
    "legacy_report_tool_used_rate": bool_rate([item["metrics"]["metrics"].get("legacy_report_tool_used") for item in all_results]),
}

payload = {
    "experiment_id": "qwen_multi_agent_batch_colab",
    "architecture": ARCHITECTURE_MODE,
    "model_name": MODEL_NAME,
    "dataset_ref": DATASET_REF,
    "dataset_path": str(DATASET_PATH),
    "model_config": {
        "model_name": MODEL_NAME,
        "load_in_4bit": LOAD_IN_4BIT,
        "load_in_8bit": LOAD_IN_8BIT,
        "torch_dtype": TORCH_DTYPE,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "do_sample": DO_SAMPLE,
    },
    "agent_config": {
        "max_orchestration_steps": MAX_ORCHESTRATION_STEPS,
        "max_role_steps": MAX_ROLE_STEPS,
        "max_tool_calls_per_role": MAX_TOOL_CALLS_PER_ROLE,
        "max_parse_retries": MAX_PARSE_RETRIES,
        "max_tool_retries": MAX_TOOL_RETRIES,
        "state_feedback_mode": STATE_FEEDBACK_MODE,
    },
    "summary": summary,
    "results": all_results,
}

with BATCH_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(to_jsonable(payload), f, ensure_ascii=False, indent=2)

print("Saved batch:", BATCH_OUTPUT_PATH)
summary_df

## 12. Inspect raw model outputs

In [ ]:
for record in all_results:
    print("=" * 100)
    print(record["selected_preset"])
    for i, raw in enumerate(record["agent_result"].get("raw_model_outputs", []), start=1):
        print("-" * 80)
        print("RAW", i)
        print(raw)

## 13. Optional architecture comparison

In [ ]:
def load_batch(path):
    if not path.exists():
        print("Missing:", path)
        return None
    return json.loads(path.read_text(encoding="utf-8"))

single_batch = load_batch(OUTPUT_DIR / "qwen_single_agent_batch_colab.json")
rule_multi_batch = load_batch(OUTPUT_DIR / "multi_agent_rule_based_five_task_batch.json")
qwen_multi_batch = load_batch(BATCH_OUTPUT_PATH)

def by_preset(payload):
    if payload is None:
        return {}
    return {item["selected_preset"]: item for item in payload.get("results", [])}

comparison_rows = []
for preset_id in sorted(set(by_preset(qwen_multi_batch)) | set(by_preset(single_batch)) | set(by_preset(rule_multi_batch))):
    single = by_preset(single_batch).get(preset_id, {})
    rule = by_preset(rule_multi_batch).get(preset_id, {})
    multi = by_preset(qwen_multi_batch).get(preset_id, {})
    comparison_rows.append({
        "preset_id": preset_id,
        "task_type": (multi.get("task_config") or single.get("task_config") or rule.get("task_config") or {}).get("task_type"),
        "qwen_single_success": (single.get("agent_result") or {}).get("success"),
        "qwen_single_overall_ok": single.get("overall_ok", single.get("success")),
        "qwen_single_tool_sequence_match": ((single.get("metrics") or {}).get("metrics") or {}).get("tool_sequence_match"),
        "qwen_single_final_answer_present": bool((single.get("agent_result") or {}).get("answer")),
        "qwen_single_parse_error_count": (single.get("agent_result") or {}).get("parse_error_count"),
        "qwen_single_stalled_loop_detected": (single.get("agent_result") or {}).get("stalled_loop_detected"),
        "rule_multi_success": (rule.get("agent_result") or {}).get("status") == "success",
        "rule_multi_overall_ok": rule.get("overall_ok"),
        "rule_multi_tool_sequence_match": ((rule.get("metrics") or {}).get("metrics") or {}).get("tool_sequence_match"),
        "rule_multi_role_order_match": ((rule.get("metrics") or {}).get("metrics") or {}).get("role_agent_order_match"),
        "rule_multi_artifact_flow_valid": ((rule.get("metrics") or {}).get("metrics") or {}).get("artifact_flow_valid"),
        "qwen_multi_success": (multi.get("agent_result") or {}).get("success"),
        "qwen_multi_overall_ok": multi.get("overall_ok"),
        "qwen_multi_tool_sequence_match": ((multi.get("metrics") or {}).get("metrics") or {}).get("tool_sequence_match"),
        "qwen_multi_role_order_match": ((multi.get("metrics") or {}).get("metrics") or {}).get("role_agent_order_match"),
        "qwen_multi_artifact_flow_valid": ((multi.get("metrics") or {}).get("metrics") or {}).get("artifact_flow_valid"),
        "qwen_multi_final_answer_present": bool((multi.get("agent_result") or {}).get("answer")),
        "qwen_multi_parse_error_count": (multi.get("agent_result") or {}).get("parse_error_count"),
        "qwen_multi_stalled_loop_detected": (multi.get("agent_result") or {}).get("stalled_loop_detected"),
    })

pd.DataFrame(comparison_rows)